## Imports and Setup

In [37]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

print("Loading raw data from Final_Cleaned_DATA.csv...")
# Adjust this path if your CSV is in a different directory
df = pd.read_csv('Final_Cleaned_DATA.csv') 

# Remove the specific clinic we are excluding from the study
clinic_to_remove = 'Aimat. Anosokatestalmena'
df = df[df['Clinic Name'] != clinic_to_remove].copy()
print(f"Removed clinic: {clinic_to_remove}")

Loading raw data from Final_Cleaned_DATA.csv...
Removed clinic: Aimat. Anosokatestalmena


## Medical Feature Engineering

In [38]:
print("Engineering standard and scientific thyroid features...")
# Map Gender to binary numerical values
df['Gender_Encoded'] = df['Gender'].map({'F': 0, 'M': 1, 'Female': 0, 'Male': 1})

# Standard Medical Ratios (adding 0.001 to prevent division by zero)
df['T3_FT4_ratio'] = df['T3'] / (df['FT4'] + 0.001)
df['T3_TSH_ratio'] = df['T3'] / (df['TSH'] + 0.001)

# Capture doctor intent (Did the physician order antibody tests?)
df['AntiTPO_Measured'] = df['Anti-TPO'].notna().astype(int)
df['AntiTG_Measured'] = df['Anti-TG'].notna().astype(int)

# --- SCIENTIFIC INDICES ---
# 1. TT4RI (Thyrotroph T4 Resistance Index)
df['TT4RI'] = df['FT4'] * df['TSH']

# 2. Peripheral Deiodinase Proxy (Kidney Conversion efficiency)
# Adjusts the T3/FT4 ratio by age-related metabolic decline
df['Deiodinase_Proxy'] = df['T3_FT4_ratio'] * (1 - (df['Age'] / 100))

# 3. Age-Adjusted TSH Deviation
# Expected TSH roughly rises by 0.025 mIU/L per year after a base of ~1.5
df['Expected_TSH'] = 1.5 + (0.025 * df['Age'])
df['TSH_Deviation'] = df['TSH'] - df['Expected_TSH']
# ------------------------------



Engineering standard and scientific thyroid features...


## Category Cleanup & Balancing

In [39]:
print("Balancing clinic classes...")
# Clean up ghost categories from the dropped clinic
df['Clinic Name'] = df['Clinic Name'].astype('category').cat.remove_unused_categories()
df['Clinic_Encoded'] = df['Clinic Name'].cat.codes

# Force all clinics to have the exact same number of patients (Undersampling)
min_size = df['Clinic Name'].value_counts().min()
df_balanced = df.groupby('Clinic Name').sample(n=min_size, random_state=42)
print(f"Dataset successfully balanced to {min_size} patients per clinic.")

Balancing clinic classes...
Dataset successfully balanced to 7836 patients per clinic.


## Define Features & Split Data

In [40]:
feature_cols = [
    'T3', 'FT4', 'TSH', 'Age', 'Gender_Encoded',
    'T3_FT4_ratio', 'T3_TSH_ratio',
    'AntiTPO_Measured', 'AntiTG_Measured',
    'TT4RI', 'Deiodinase_Proxy', 'TSH_Deviation'
]

X = df_balanced[feature_cols]
y = df_balanced['Clinic_Encoded']

# Split 80% for training, 20% for the final evaluation (testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Handle Missing Values (Two Versions)

#### While some Models accept the Raw data with the missing values, more traditional ones like RandomForrest don't, so we make two separate sets, one with the missing data (Raw) and one using the median to fill.

In [41]:
print("Creating two versions of the dataset (Imputed and Raw)...")

# Version A: The RAW data (NaNs remain intact). 
# We keep this for modern, sparsity-aware models like HistBoost and XGBoost.
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

# Version B: The IMPUTED data.
# We fill NaNs with medians for older, stricter models like Random Forest.
imputer = SimpleImputer(strategy='median')
# Fit only on training data to prevent data leakage, then transform both
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

Creating two versions of the dataset (Imputed and Raw)...


## Save BOTH Datasets for the Models

In [42]:
os.makedirs('data', exist_ok=True)

print("Saving cleaned datasets to the 'data/' folder...")

# Save the Target variables (y is the same for all models)
y_train.to_csv('data/y_train_clean.csv', index=False)
y_test.to_csv('data/y_test_clean.csv', index=False)

# Save Version A (For HistBoost / XGBoost)
X_train_raw.to_csv('data/X_train_clean_raw.csv', index=False)
X_test_raw.to_csv('data/X_test_clean_raw.csv', index=False)

# Save Version B (For Random Forest / SVM)
X_train_imputed.to_csv('data/X_train_clean_imputed.csv', index=False)
X_test_imputed.to_csv('data/X_test_clean_imputed.csv', index=False)

print("Preprocessing Complete!")

Saving cleaned datasets to the 'data/' folder...
Preprocessing Complete!


## Validation

In [43]:
# Run this BEFORE your K-Fold loop to clean the data once and for all
df_clean = df_balanced[df_balanced['Clinic_Encoded'].isin([0, 1])].copy()
X_train_raw = df_clean[feature_cols] # Update your X_train_raw to this
y_train = df_clean['Clinic_Encoded']

